# Measuring tone with financial dictionaries

The original run produced 100 filing rows and eight columns.

**Running in Google Colab.** Point `downloadDir` to the SEC filing text files and `wordListDir` to PositiveWords.txt, NegativeWords.txt, and LitigiousWords.txt. Run the cells in order.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import nltk
import pandas as pd
from bs4 import BeautifulSoup
from nltk.tokenize import word_tokenize
from nltk.probability import FreqDist

nltk.download('punkt')
nltk.download('punkt_tab')

downloadDir = "/content/drive/MyDrive/TextMining/My10-Ks/"   # 10-Ks downloaded last week
wordListDir = "/content/drive/MyDrive/TextMining/WordLists/"   # Positive/Negative/Litigious txt from LM dictionary

tenKFiles = sorted(os.listdir(downloadDir))[:100]
print(len(tenKFiles))
print(tenKFiles[:5])

In [ ]:
with open(downloadDir + tenKFiles[0]) as tenK:
  html = tenK.read()

soup = BeautifulSoup(html)
cleaned = soup.get_text()

#print(cleaned[10000:15000])

textList = word_tokenize(cleaned.lower())   #nltk word parser, 'company,' -> 'company' ','
textList = [word for word in textList if word.isalpha()]   #drop punctuation and numbers

wordDict = FreqDist(textList)   # {'the': 3495, 'company': 434 ...}

totalWords = 0
totalWords = len(textList)

print(len(wordDict.keys()))
print(totalWords)
print(wordDict.get('company', 0))
print(wordDict.get('company,', 0))   #0 now
print(wordDict.most_common(50))

In [ ]:
with open(wordListDir + "PositiveWords.txt") as positiveW:
  positiveWords = positiveW.read().lower().split()

with open(wordListDir + "NegativeWords.txt") as negativeW:
  negativeWords = negativeW.read().lower().split()

with open(wordListDir + "LitigiousWords.txt") as litigiousW:
  litigiousWords = litigiousW.read().lower().split()

print(len(positiveWords), len(negativeWords), len(litigiousWords))
print(negativeWords[:20])

In [ ]:
positiveWordCount = 0
negativeWordCount = 0
litigiousWordCount = 0

positivityRate = 0.0
negativityRate = 0.0
litigiousRate = 0.0

for word in wordDict:
  if word in positiveWords:
    positiveWordCount += wordDict[word]
  if word in negativeWords:
    negativeWordCount += wordDict[word]
  if word in litigiousWords:
    litigiousWordCount += wordDict[word]

positivityRate = positiveWordCount / totalWords
negativityRate = negativeWordCount / totalWords
litigiousRate = litigiousWordCount / totalWords

print(tenKFiles[0])
print(totalWords)
print(positiveWordCount, positivityRate)
print(negativeWordCount, negativityRate)
print(litigiousWordCount, litigiousRate)

In [ ]:
positiveWords = set(positiveWords)     #set makes 'in' much faster for 100 files
negativeWords = set(negativeWords)
litigiousWords = set(litigiousWords)

results = []   # one dict per 10-K -> rows of the dataframe

count = 0
for fileName in tenKFiles:

  with open(downloadDir + fileName) as tenK:
    html = tenK.read()

  soup = BeautifulSoup(html)
  cleaned = soup.get_text()

  textList = word_tokenize(cleaned.lower())
  textList = [word for word in textList if word.isalpha()]

  wordDict = FreqDist(textList)
  totalWords = len(textList)

  positiveWordCount = 0
  negativeWordCount = 0
  litigiousWordCount = 0

  for word in wordDict:
    if word in positiveWords:
      positiveWordCount += wordDict[word]
    if word in negativeWords:
      negativeWordCount += wordDict[word]
    if word in litigiousWords:
      litigiousWordCount += wordDict[word]

  results.append({'filename': fileName,
                  'wordCount': totalWords,
                  'PositiveWordCount': positiveWordCount,
                  'NegativeWordCount': negativeWordCount,
                  'LitigiousWordCount': litigiousWordCount,
                  'PositivityRate': positiveWordCount / totalWords,
                  'NegativityRate': negativeWordCount / totalWords,
                  'LitigiousRate': litigiousWordCount / totalWords})

  count += 1
  print(count, fileName, totalWords)

df = pd.DataFrame(results)
df

In [ ]:
print(df.shape)
df.describe()